# Deep Agents: Building Complex Agents for Long-Horizon Tasks

In this notebook, we'll explore **Deep Agents** - a new approach to building AI agents that can handle complex, multi-step tasks over extended periods. We'll implement all four key elements of Deep Agents while building on our Personal Wellness Assistant use case.

**Learning Objectives:**
- Understand the four key elements of Deep Agents: Planning, Context Management, Subagent Spawning, and Long-term Memory
- Implement each element progressively using the `deepagents` package
- Learn to use Skills for progressive capability disclosure
- Use the `deepagents-cli` for interactive agent sessions

## Table of Contents:

- **Breakout Room #1:** Deep Agent Foundations
  - Task 1: Dependencies & Setup
  - Task 2: Understanding Deep Agents
  - Task 3: Planning with Todo Lists
  - Task 4: Context Management with File Systems
  - Task 5: Basic Deep Agent
  - Question #1 & Question #2
  - Activity #1: Build a Research Agent

- **Breakout Room #2:** Advanced Features & Integration
  - Task 6: Subagent Spawning
  - Task 7: Long-term Memory Integration
  - Task 8: Skills - On-Demand Capabilities
  - Task 9: Using deepagents-cli
  - Task 10: Building a Complete Deep Agent System
  - Question #3 & Question #4
  - Activity #2: Build a Wellness Coach Agent

---
# 🤝 Breakout Room #1
## Deep Agent Foundations

## Task 1: Dependencies & Setup

Before we begin, make sure you have:

1. **API Keys** for:
   - Anthropic (default for Deep Agents) or OpenAI
   - LangSmith (optional, for tracing)
   - Tavily (optional, for web search)

2. **Dependencies installed** via `uv sync`

3. **For the CLI** (Task 9): `uv pip install deepagents-cli`

### Environment Setup

You can either:
- Create a `.env` file with your API keys (recommended):
  ```
  ANTHROPIC_API_KEY=your_key_here
  OPENAI_API_KEY=your_key_here
  LANGCHAIN_API_KEY=your_key_here
  ```
- Or enter them interactively when prompted

In [1]:
# Core imports
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

def get_api_key(env_var: str, prompt: str) -> str:
    """Get API key from environment or prompt user."""
    value = os.environ.get(env_var, "")
    if not value:
        value = getpass.getpass(prompt)
        if value:
            os.environ[env_var] = value
    return value

In [2]:
# Set Anthropic API Key (default for Deep Agents)
anthropic_key = get_api_key("ANTHROPIC_API_KEY", "Anthropic API Key: ")
if anthropic_key:
    print("Anthropic API key set")
else:
    print("Warning: No Anthropic API key configured")

Anthropic API key set


In [3]:
# Optional: OpenAI for alternative models and subagents
openai_key = get_api_key("OPENAI_API_KEY", "OpenAI API Key (press Enter to skip): ")
if openai_key:
    print("OpenAI API key set")
else:
    print("OpenAI API key not configured (optional)")

OpenAI API key set


In [4]:
# Optional: LangSmith for tracing
langsmith_key = get_api_key("LANGCHAIN_API_KEY", "LangSmith API Key (press Enter to skip): ")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Deep Agents - {uuid4().hex[0:8]}"
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")

LangSmith tracing enabled. Project: AIE9 - Deep Agents - 464bfc7a


In [5]:
# Verify deepagents installation
from deepagents import create_deep_agent
print("deepagents package imported successfully!")

# Test with a simple agent
test_agent = create_deep_agent()
result = test_agent.invoke({
    "messages": [{"role": "user", "content": "Say 'Deep Agents ready!' in exactly those words."}]
})
print(result["messages"][-1].content)

deepagents package imported successfully!
Deep Agents ready!


## Task 2: Understanding Deep Agents

**Deep Agents** represent a shift from simple tool-calling loops to sophisticated agents that can handle complex, long-horizon tasks. They address four key challenges:

### The Four Key Elements

| Element | Challenge Addressed | Implementation |
|---------|---------------------|----------------|
| **Planning** | "What should I do?" | Todo lists that persist task state |
| **Context Management** | "What do I know?" | File systems for storing/retrieving info |
| **Subagent Spawning** | "Who can help?" | Task tool for delegating to specialists |
| **Long-term Memory** | "What did I learn?" | LangGraph Store for cross-session memory |

### Deep Agents vs Traditional Agents

```
Traditional Agent Loop:
┌─────────────────────────────────────┐
│  User Query                         │
│       ↓                             │
│  Think → Act → Observe → Repeat     │
│       ↓                             │
│  Response                           │
└─────────────────────────────────────┘
Problems: Context bloat, no delegation,
          loses track of complex tasks

Deep Agent Architecture:
┌─────────────────────────────────────────────────────────┐
│                    Deep Agent                           │
├─────────────────────────────────────────────────────────┤
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │   PLANNING   │  │   CONTEXT    │  │   MEMORY     │   │
│  │              │  │  MANAGEMENT  │  │              │   │
│  │ write_todos  │  │              │  │   Store      │   │
│  │ update_todo  │  │  read_file   │  │  namespace   │   │
│  │ list_todos   │  │  write_file  │  │  get/put     │   │
│  │              │  │  edit_file   │  │              │   │
│  └──────────────┘  │  ls          │  └──────────────┘   │
│                    └──────────────┘                     │
│  ┌──────────────────────────────────────────────────┐   │
│  │              SUBAGENT SPAWNING                   │   │
│  │                                                  │   │
│  │  task(prompt, tools, model, system_prompt)       │   │
│  │       ↓              ↓              ↓            │   │
│  │  ┌────────┐    ┌────────┐    ┌────────┐          │   │
│  │  │Research│    │Writing │    │Analysis│          │   │
│  │  │Subagent│    │Subagent│    │Subagent│          │   │
│  │  └────────┘    └────────┘    └────────┘          │   │
│  └──────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

### When to Use Deep Agents

| Use Case | Traditional Agent | Deep Agent |
|----------|-------------------|------------|
| Simple Q&A | ✅ | Overkill |
| Single-step tool use | ✅ | Overkill |
| Multi-step research | ⚠️ May lose track | ✅ |
| Complex projects | ❌ Context overflow | ✅ |
| Parallel task execution | ❌ | ✅ |
| Long-running sessions | ❌ | ✅ |

### Key Insight: "Planning is Context Engineering"

Deep Agents treat planning not as a separate phase, but as **context engineering**:
- Todo lists aren't just task trackers—they're **persistent context** about what to do
- File systems aren't just storage—they're **extended memory** beyond the context window
- Subagents aren't just helpers—they're **context isolation** to prevent bloat

## Task 3: Planning with Todo Lists

The first key element of Deep Agents is **Planning**. Instead of trying to hold all task state in the conversation, Deep Agents use structured todo lists.

### Why Todo Lists?

1. **Persistence**: Tasks survive across conversation turns
2. **Visibility**: Both agent and user can see progress
3. **Structure**: Clear tracking of what's done vs pending
4. **Recovery**: Agent can resume from where it left off

### Todo List Tools

| Tool | Purpose |
|------|----------|
| `write_todos` | Create a structured task list |
| `update_todo` | Mark tasks as complete/in-progress |
| `list_todos` | View current task state |

In [6]:
from langchain_core.tools import tool
from typing import List, Optional
import json

# Simple in-memory todo storage for demonstration
# In production, Deep Agents use persistent storage
TODO_STORE = {}

@tool
def write_todos(todos: List[dict]) -> str:
    """Create a list of todos for tracking task progress.
    
    Args:
        todos: List of todo items, each with 'title' and optional 'description'
    
    Returns:
        Confirmation message with todo IDs
    """
    created = []
    for i, todo in enumerate(todos):
        todo_id = f"todo_{len(TODO_STORE) + i + 1}"
        TODO_STORE[todo_id] = {
            "id": todo_id,
            "title": todo.get("title", "Untitled"),
            "description": todo.get("description", ""),
            "status": "pending"
        }
        created.append(todo_id)
    return f"Created {len(created)} todos: {', '.join(created)}"

@tool
def update_todo(todo_id: str, status: Literal["pending", "in_progress", "completed"]) -> str:
    """Update the status of a todo item.
    
    Args:
        todo_id: The ID of the todo to update
        status: New status (pending, in_progress, completed)
    
    Returns:
        Confirmation message
    """
    if todo_id not in TODO_STORE:
        return f"Todo {todo_id} not found"
    TODO_STORE[todo_id]["status"] = status
    return f"Updated {todo_id} to {status}"

@tool
def list_todos() -> str:
    """List all todos with their current status.
    
    Returns:
        Formatted list of all todos
    """
    if not TODO_STORE:
        return "No todos found"
    
    result = []
    for todo_id, todo in TODO_STORE.items():
        status_emoji = {"pending": "⬜", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result.append(f"{emoji} [{todo_id}] {todo['title']} ({todo['status']})")
    return "\n".join(result)

print("Todo tools defined!")

Todo tools defined!


In [7]:
# Test the todo tools
TODO_STORE.clear()  # Reset for demo

# Create some wellness todos
result = write_todos.invoke({
    "todos": [
        {"title": "Assess current sleep patterns", "description": "Review user's sleep schedule and quality"},
        {"title": "Research sleep improvement strategies", "description": "Find evidence-based techniques"},
        {"title": "Create personalized sleep plan", "description": "Combine findings into actionable steps"},
    ]
})
print(result)
print("\nCurrent todos:")
print(list_todos.invoke({}))

Created 3 todos: todo_1, todo_3, todo_5

Current todos:
⬜ [todo_1] Assess current sleep patterns (pending)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


In [8]:
# Simulate progress
update_todo.invoke({"todo_id": "todo_1", "status": "completed"})
update_todo.invoke({"todo_id": "todo_2", "status": "in_progress"})

print("After updates:")
print(list_todos.invoke({}))

After updates:
✅ [todo_1] Assess current sleep patterns (completed)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


## Task 4: Context Management with File Systems

The second key element is **Context Management**. Deep Agents use file systems to:

1. **Offload large content** - Store research, documents, and results to disk
2. **Persist across sessions** - Files survive beyond conversation context
3. **Share between subagents** - Subagents can read/write shared files
4. **Prevent context overflow** - Large tool results automatically saved to disk

### Automatic Context Management

Deep Agents automatically handle context limits:
- **Large result offloading**: Tool results >20k tokens → saved to disk
- **Proactive offloading**: At 85% context capacity → agent saves state to disk
- **Summarization**: Long conversations get summarized while preserving intent

### File System Tools

| Tool | Purpose |
|------|----------|
| `ls` | List directory contents |
| `read_file` | Read file contents |
| `write_file` | Create/overwrite files |
| `edit_file` | Make targeted edits |

In [9]:
import os
from pathlib import Path

# Create a workspace directory for our agent
WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

@tool
def ls(path: str = ".") -> str:
    """List contents of a directory.
    
    Args:
        path: Directory path to list (default: current directory)
    
    Returns:
        List of files and directories
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"Directory not found: {path}"
    
    items = []
    for item in sorted(target.iterdir()):
        prefix = "[DIR]" if item.is_dir() else "[FILE]"
        size = f" ({item.stat().st_size} bytes)" if item.is_file() else ""
        items.append(f"{prefix} {item.name}{size}")
    
    return "\n".join(items) if items else "(empty directory)"

@tool
def read_file(path: str) -> str:
    """Read contents of a file.
    
    Args:
        path: Path to the file to read
    
    Returns:
        File contents
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    return target.read_text()

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file (creates or overwrites).
    
    Args:
        path: Path to the file to write
        content: Content to write to the file
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}"

@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing text.
    
    Args:
        path: Path to the file to edit
        old_text: Text to find and replace
        new_text: Replacement text
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    
    content = target.read_text()
    if old_text not in content:
        return f"Text not found in {path}"
    
    new_content = content.replace(old_text, new_text, 1)
    target.write_text(new_content)
    return f"Updated {path}"

print("File system tools defined!")
print(f"Workspace: {WORKSPACE.absolute()}")

File system tools defined!
Workspace: c:\Users\Manish Kumar\ai-bootcamp\Interactive-Dev-Environment-for-AI-Engineers\AIE9\07_Deep_Agents\workspace


In [10]:
# Test the file system tools
print("Current workspace contents:")
print(ls.invoke({"path": "."}))

Current workspace contents:
[FILE] comprehensive_sleep_research_summary.md (20744 bytes)
[FILE] evidence_based_stress_management_guide.md (13741 bytes)
[FILE] my_sleep_improvement_plan.md (7420 bytes)
[FILE] personalized_sleep_improvement_plan.md (6840 bytes)
[DIR] research
[FILE] stress_management_research_report.md (113883 bytes)


In [11]:
# Create a research notes file
notes = """# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations
"""

result = write_file.invoke({"path": "research/sleep_notes.md", "content": notes})
print(result)

# Verify it was created
print("\nResearch directory:")
print(ls.invoke({"path": "research"}))

Wrote 242 characters to research/sleep_notes.md

Research directory:
[FILE] actionable_sleep_improvement_plan.md (8520 bytes)
[FILE] sleep_improvement_evidence_based_strategies.md (12298 bytes)
[FILE] sleep_notes.md (252 bytes)


In [12]:
# Read and edit the file
print("File contents:")
print(read_file.invoke({"path": "research/sleep_notes.md"}))

File contents:
# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations



## Task 5: Basic Deep Agent

Now let's create a basic Deep Agent using the `deepagents` package. This combines:
- Planning (todo lists)
- Context management (file system)
- A capable LLM backbone

### Configuring the FilesystemBackend

Deep Agents come with **built-in file tools** (`ls`, `read_file`, `write_file`, `edit_file`). To control where files are stored, we configure a `FilesystemBackend`:

```python
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(
    root_dir="/path/to/workspace",
    virtual_mode=True  # REQUIRED to actually sandbox files!
)
```

**Critical: `virtual_mode=True`**
- Without `virtual_mode=True`, agents can still write anywhere on the filesystem!
- The `root_dir` alone does NOT restrict file access
- `virtual_mode=True` blocks paths with `..`, `~`, and absolute paths outside root

In [13]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Configure the filesystem backend to use our workspace directory
# IMPORTANT: virtual_mode=True is required to actually restrict paths to root_dir
# Without it, agents can still write anywhere on the filesystem!
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Combine our custom tools (for todo tracking)
# Note: Deep Agents has built-in file tools (ls, read_file, write_file, edit_file)
# that will use the configured FilesystemBackend
custom_tools = [
    write_todos,
    update_todo,
    list_todos,
]

# Create a basic Deep Agent
wellness_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=custom_tools,
    backend=filesystem_backend,  # Configure where files are stored
    system_prompt="""You are a Personal Wellness Assistant that helps users improve their health.

When given a complex task:
1. First, create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear summary when complete

Be thorough but concise. Always explain your reasoning."""
)

print(f"Basic Deep Agent created!")
print(f"File operations sandboxed to: {workspace_path}")

Basic Deep Agent created!
File operations sandboxed to: c:\Users\Manish Kumar\ai-bootcamp\Interactive-Dev-Environment-for-AI-Engineers\AIE9\07_Deep_Agents\workspace


In [14]:
# Reset todo store for fresh demo
TODO_STORE.clear()

# Test with a multi-step wellness task
result = wellness_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please create a personalized sleep improvement plan for me and save it to a file."""
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Perfect! I've created your personalized sleep improvement plan and saved it to `/your_sleep_transformation_plan.md`. Let me give you a quick summary of what I've prepared for you:

## 🌙 Your Personalized Sleep Plan is Ready!

### **Key Features of Your Plan:**

1. **30-Day Roadmap** - Week-by-week progression tailored to your specific issues
2. **Targeted Solutions** for each of your challenges:
   - Inconsistent bedtimes → Gradual shifting strategy with consistency tricks
   - Phone in bed → Physical separation + replacement habits
   - Morning fatigue → Optimized wake routine + circadian rhythm regulation

3. **Start-Today Actions:**
   - Fix wake time at 7:00 AM (most important!)
   - Move phone charger outside bedroom tonight
   - Get bright light within 10 minutes of waking tomorrow

### **Your Realistic Timeline:**
- **Week 1:** Foundation building (expect adjustment period)
- **Week 2:** Routine establishment (sleep gets easier)  
- **Week 3:** Quality improvemen

In [15]:
# Check what the agent created
print("Todo list after task:")
print(list_todos.invoke({}))

print("\n" + "="*50)
print("\nWorkspace contents:")
# List files in the workspace directory
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [DIR] {f.name}/")

Todo list after task:
✅ [todo_1] Analyze current sleep issues (completed)
✅ [todo_3] Research evidence-based sleep improvement strategies (completed)
✅ [todo_5] Research sleep hygiene best practices (completed)
✅ [todo_7] Save plan to file for reference (completed)
✅ [todo_6] Investigate circadian rhythm regulation techniques (completed)
✅ [todo_8] Analyze screen time impact and mitigation strategies (completed)
✅ [todo_10] Research sleep schedule consistency methods (completed)
✅ [todo_12] Investigate morning energy optimization techniques (completed)
✅ [todo_14] Compile comprehensive research summary (completed)


Workspace contents:
  [FILE] actionable_sleep_plan.md (6977 bytes)
  [FILE] comprehensive_sleep_research_summary.md (20744 bytes)
  [FILE] evidence_based_stress_management_guide.md (13741 bytes)
  [FILE] my_sleep_improvement_plan.md (7420 bytes)
  [FILE] personalized_sleep_improvement_plan.md (6840 bytes)
  [DIR] research/
  [FILE] sleep_research.md (8514 bytes)
  [FILE] st

---
## ❓ Question #1:

What are the **trade-offs** of using todo lists for planning? Consider:
- When might explicit planning overhead slow things down?
- How granular should todo items be?
- What happens if the agent creates todos but never completes them?

##### Answer:
Using todo lists for planning is helpful, but it’s not always the best choice. If the task is small or straightforward, making a todo list first can actually slow things down because the agent spends time planning instead of just doing the work. Planning makes more sense when the task has multiple steps or needs coordination over time.

Todo items also shouldn’t be too detailed. If they’re too granular, like breaking every tiny action into a task, the list becomes noisy and hard to follow. On the other hand, if the todos are too broad, they don’t really guide the agent. A good balance is when each todo represents one meaningful step.

Another issue is when the agent keeps creating todos but never completes them. In that case, the agent looks like it’s busy but isn’t actually making progress. Over time, unfinished todos can clutter the context and make the agent unreliable. So todo lists are useful, but only when they’re used carefully and actually executed.

## ❓ Question #2:

How would you design a **context management strategy** for a wellness agent that:
- Needs to reference a large health document (16KB)
- Tracks user metrics over time
- Must remember user conditions (allergies, medications) for safety

What goes in files vs. in the prompt? What should never be offloaded?

##### Answer:
For a wellness agent, context management needs to be split based on what kind of information we’re dealing with.

- The large health document shouldn’t live in the prompt because it’s too big and mostly reference material. It makes more sense to store it in files or a vector store and only pull in the small parts that are relevant to the current question.
- User metrics like sleep, steps, or mood change over time and keep growing, so they shouldn’t be kept in the prompt either. Those should be stored separately and summarized when needed. The agent doesn’t need every data point, just trends or recent averages.
 - User conditions like allergies and medications are different. These are safety-critical and should always be present in the prompt and memory. The agent must never forget them, because missing this information could lead to unsafe responses.

In short, big static knowledge goes in files, growing data goes in storage with summaries, and safety-related information stays in the prompt. Anything related to user safety should never be offloaded or made optional.

---
## 🏗️ Activity #1: Build a Research Agent

Build a Deep Agent that can research a wellness topic and produce a structured report.

### Requirements:
1. Create todos for the research process
2. Read from the HealthWellnessGuide.txt in the data folder
3. Save findings to a structured markdown file
4. Update todo status as tasks complete

### Test prompt:
"Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."

In [16]:
### YOUR CODE HERE ###

# Step 0: Reset todo store for clean run
TODO_STORE.clear()


# Step 1: Create a research agent with appropriate tools
# (We reuse the same backend and tool pattern from the notebook)

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model
from pathlib import Path

workspace_path = Path("workspace").absolute()

filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True
)

custom_tools = [
    write_todos,
    update_todo,
    list_todos,
]

research_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=custom_tools,
    backend=filesystem_backend,
    system_prompt="""
You are a wellness research agent.

When given a research task:
1. First create a todo list using write_todos.
2. Read HealthWellnessGuide.txt from the data folder if needed.
3. Work through each todo and update its status using update_todo.
4. Save the final structured markdown guide using write_file.
5. Provide a short summary when complete.

Be structured, concise, and follow the steps clearly.
"""
)


# Step 2: Add a tool to read from the data folder
# (We define it exactly using @tool as in notebook style)

from langchain_core.tools import tool

@tool
def read_wellness_guide() -> str:
    """Read the HealthWellnessGuide.txt file from the data folder."""
    path = Path("data/HealthWellnessGuide.txt")
    if not path.exists():
        return "HealthWellnessGuide.txt not found."
    return path.read_text()

# Re-create agent including this new tool
custom_tools.append(read_wellness_guide)

research_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=custom_tools,
    backend=filesystem_backend,
    system_prompt="""
You are a wellness research agent.

When given a research task:
1. First create a todo list using write_todos.
2. Use read_wellness_guide to review the wellness content.
3. Work through each todo and update its status using update_todo.
4. Save the final structured markdown guide using write_file.
5. Provide a short summary when complete.

Be structured, concise, and follow the steps clearly.
"""
)


# Step 4: Test with the stress management research task

result = research_agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Research stress management techniques and create a comprehensive guide "
            "with at least 5 evidence-based strategies. "
            "Save the guide as stress_management_guide.md"
        )
    }]
})

print("Agent response:\n")
print(result["messages"][-1].content)


# Verification Section (important for instructor visibility)

print("\nTodo List After Execution:")
print(list_todos.invoke({}))

print("\nWorkspace Contents:")
print(ls.invoke({"path": "."}))


Agent response:

Perfect! I've successfully completed the comprehensive stress management guide. Here's a summary of what I accomplished:

## Summary

I've created a comprehensive, evidence-based stress management guide saved as `stress_management_guide.md` that includes:

### ✅ **5+ Evidence-Based Strategies:**
1. **Mindfulness-Based Stress Reduction (MBSR)** - 50-70% anxiety reduction
2. **Cognitive Behavioral Therapy (CBT) techniques** - 40-60% stress symptom reduction  
3. **Heart Rate Variability (HRV) Biofeedback** - 24% stress symptom reduction
4. **Expressive Writing Therapy** - Significant improvement in intrusive thoughts
5. **Social Support Network Intervention** - 50% reduction in stress-related illness
6. **Progressive Muscle Relaxation** - 42% reduction in stress symptoms
7. **Advanced breathing techniques** - 23% cortisol reduction

### ✅ **Comprehensive Structure:**
- Understanding of stress and symptoms
- Immediate relief techniques (2-5 minutes)
- Long-term evidence-b

### Activity 1 Summary

In this activity, we built a research-focused Deep Agent that follows a clear workflow instead of just generating an answer. We created explicit todos for the research process, updated their status as tasks were completed, used the wellness guide file as reference, and saved the final output as a structured markdown report. The goal was not just to generate a stress management guide, but to demonstrate proper tool usage, task tracking, and file handling using Deep Agents.

Output: As you can see in the output, all todos were marked as completed and the stress management guide was successfully saved as a markdown file in the workspace, which confirms that the agent followed the required process correctly.

---
# 🤝 Breakout Room #2
## Advanced Features & Integration

## Task 6: Subagent Spawning

The third key element is **Subagent Spawning**. This allows a Deep Agent to delegate tasks to specialized subagents.

### Why Subagents?

1. **Context Isolation**: Each subagent has its own context window, preventing bloat
2. **Specialization**: Different subagents can have different tools/prompts
3. **Parallelism**: Multiple subagents can work simultaneously
4. **Cost Optimization**: Use cheaper models for simpler subtasks

### How Subagents Work

```
Main Agent
    ├── task("Research sleep science", model="gpt-4o-mini")
    │       └── Returns: Summary of findings
    │
    ├── task("Analyze user's sleep data", tools=[analyze_tool])
    │       └── Returns: Analysis results
    │
    └── task("Write recommendations", system_prompt="Be concise")
            └── Returns: Final recommendations
```

Key benefit: The main agent only receives **summaries**, not all the intermediate context!

In [17]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Define specialized subagent configurations
# Note: Subagents inherit the backend from the parent agent
research_subagent = {
    "name": "research-agent",
    "description": "Use this agent to research wellness topics in depth. It can read documents and synthesize information.",
    "system_prompt": """You are a wellness research specialist. Your job is to:
1. Find relevant information in provided documents
2. Synthesize findings into clear summaries
3. Cite sources when possible

Be thorough but concise. Focus on evidence-based information.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model for research
}

writing_subagent = {
    "name": "writing-agent",
    "description": "Use this agent to create well-structured documents, plans, and guides.",
    "system_prompt": """You are a wellness content writer. Your job is to:
1. Take research findings and turn them into clear, actionable content
2. Structure information for easy understanding
3. Use formatting (headers, bullets, etc.) effectively

Write in a supportive, encouraging tone.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "anthropic:claude-sonnet-4-20250514",
}

print("Subagent configurations defined!")

Subagent configurations defined!


In [18]:
# Create a coordinator agent that can spawn subagents
coordinator_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[write_todos, update_todo, list_todos],
    backend=filesystem_backend,  # Use the same backend - subagents inherit it
    subagents=[research_subagent, writing_subagent],
    system_prompt="""You are a Wellness Project Coordinator. Your role is to:
1. Break down complex wellness requests into subtasks
2. Delegate research to the research-agent
3. Delegate content creation to the writing-agent
4. Coordinate the overall workflow using todos

Use subagents for specialized work rather than doing everything yourself.
This keeps the work organized and the results high-quality."""
)

print("Coordinator agent created with subagent capabilities!")

Coordinator agent created with subagent capabilities!


In [19]:
# Reset for demo
TODO_STORE.clear()

# Test the coordinator with a complex task
result = coordinator_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Create a comprehensive morning routine guide for better energy.
        
The guide should:
1. Research the science behind morning routines
2. Include practical steps for exercise, nutrition, and mindset
3. Be saved as a well-formatted markdown file"""
    }]
})

print("Coordinator response:")
print(result["messages"][-1].content)

Coordinator response:
Perfect! I've successfully created a comprehensive morning routine guide for better energy. Here's what I've delivered:

## 📋 Project Complete! 

Your **comprehensive morning routine guide** has been created and saved as `/comprehensive_morning_routine_guide.md`. The guide includes everything you requested and more:

### ✅ **Research-Based Foundation**
- Scientific evidence on circadian rhythms and energy optimization
- Studies on morning light exposure, exercise timing, and cognitive performance
- Evidence-based recommendations for habit formation

### ✅ **Practical Implementation Sections**
- **Exercise**: Multiple options from high-intensity to gentle movement with specific timing recommendations
- **Nutrition**: Hydration strategies, energy-boosting foods, and meal timing guidelines  
- **Mindset**: Gratitude practices, meditation techniques, intention-setting, and affirmations

### ✅ **Ready-to-Use Resources**
- **3 sample timelines** (30, 45, and 60 minutes)

In [20]:
# Check the results
print("Final todo status:")
print(list_todos.invoke({}))

print("\nGenerated files in workspace:")
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

Final todo status:
✅ [todo_1] Research the science behind morning routines (completed)
✅ [todo_3] Create comprehensive morning routine guide (completed)
✅ [todo_5] Save the guide as markdown file (completed)

Generated files in workspace:
  [FILE] actionable_sleep_plan.md (6977 bytes)
  [FILE] comprehensive_morning_routine_guide.md (19615 bytes)
  [FILE] comprehensive_sleep_research_summary.md (20744 bytes)
  [FILE] evidence_based_stress_management_guide.md (13741 bytes)
  [FILE] Evidence_Based_Stress_Management_Report.md (20303 bytes)
  [FILE] morning_routine_energy_guide.md (67791 bytes)
  [FILE] my_sleep_improvement_plan.md (7420 bytes)
  [FILE] personalized_sleep_improvement_plan.md (6840 bytes)
  [DIR] research/
  [FILE] sleep_research.md (8514 bytes)
  [FILE] stress_management_guide.md (11193 bytes)
  [FILE] stress_management_research_report.md (113883 bytes)
  [FILE] your_sleep_transformation_plan.md (6506 bytes)


## Task 7: Long-term Memory Integration

The fourth key element is **Long-term Memory**. Deep Agents integrate with LangGraph's Store for persistent memory across sessions.

### Memory Types in Deep Agents

| Type | Scope | Use Case |
|------|-------|----------|
| **Thread Memory** | Single conversation | Current session context |
| **User Memory** | Across threads, per user | User preferences, history |
| **Shared Memory** | Across all users | Common knowledge, learned patterns |

### Integration with LangGraph Store

Deep Agents can use the same `InMemoryStore` (or `PostgresStore`) we learned in Session 6:

In [21]:
from langgraph.store.memory import InMemoryStore

# Create a memory store
memory_store = InMemoryStore()

# Store user profile
user_id = "user_alex"
profile_namespace = (user_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Alex"})
memory_store.put(profile_namespace, "goals", {
    "primary": "improve energy levels",
    "secondary": "better sleep"
})
memory_store.put(profile_namespace, "conditions", {
    "dietary": ["vegetarian"],
    "medical": ["mild anxiety"]
})
memory_store.put(profile_namespace, "preferences", {
    "exercise_time": "morning",
    "communication_style": "detailed"
})

print(f"Stored profile for {user_id}")

# Retrieve and display
for item in memory_store.search(profile_namespace):
    print(f"  {item.key}: {item.value}")

Stored profile for user_alex
  name: {'value': 'Alex'}
  goals: {'primary': 'improve energy levels', 'secondary': 'better sleep'}
  conditions: {'dietary': ['vegetarian'], 'medical': ['mild anxiety']}
  preferences: {'exercise_time': 'morning', 'communication_style': 'detailed'}


In [22]:
# Create memory-aware tools
from langgraph.store.base import BaseStore

@tool
def get_user_profile(user_id: str) -> str:
    """Retrieve a user's wellness profile from long-term memory.
    
    Args:
        user_id: The user's unique identifier
    
    Returns:
        User profile as formatted text
    """
    namespace = (user_id, "profile")
    items = list(memory_store.search(namespace))
    
    if not items:
        return f"No profile found for {user_id}"
    
    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)

@tool
def save_user_preference(user_id: str, key: str, value: str) -> str:
    """Save a user preference to long-term memory.
    
    Args:
        user_id: The user's unique identifier
        key: The preference key
        value: The preference value
    
    Returns:
        Confirmation message
    """
    namespace = (user_id, "preferences")
    memory_store.put(namespace, key, {"value": value})
    return f"Saved preference '{key}' for {user_id}"

print("Memory tools defined!")

Memory tools defined!


In [23]:
# Create a memory-enhanced agent
memory_tools = [
    get_user_profile,
    save_user_preference,
    write_todos,
    update_todo,
    list_todos,
]

memory_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=memory_tools,
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a Personal Wellness Assistant with long-term memory.

At the start of each conversation:
1. Check the user's profile to understand their goals and conditions
2. Personalize all advice based on their profile
3. Save any new preferences they mention

Always reference stored information to show you remember the user."""
)

print("Memory-enhanced agent created!")

Memory-enhanced agent created!


In [24]:
# Test the memory agent
TODO_STORE.clear()

result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi! My user_id is user_alex. What exercise routine would you recommend for me?"
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Hi Alex! Great to see you again. Based on your profile, I remember that your primary goal is to improve energy levels and you want better sleep as well. I also see you prefer morning workouts and have mild anxiety, so I'll tailor my recommendations accordingly.

Here's a personalized exercise routine that should help boost your energy and improve your sleep quality:

## **Morning Energy-Boosting Routine (20-30 minutes)**

### **Monday, Wednesday, Friday: Energizing Cardio + Strength**
1. **5-minute gentle warm-up**: Light stretching or walking in place
2. **15 minutes moderate cardio**: 
   - Brisk walking, cycling, or dancing (great for energy without being too intense for anxiety)
   - Keep intensity at 6-7/10 - you should be able to hold a conversation
3. **8-10 minutes bodyweight strength**:
   - Push-ups (modified if needed)
   - Squats
   - Plank hold (30-60 seconds)
   - Lunges
4. **2-3 minutes cool-down**: Deep breathing and gentle stretching

### **Tuesday, Thu

## Task 8: Skills - On-Demand Capabilities

**Skills** are a powerful feature for progressive capability disclosure. Instead of loading all tools upfront, agents can load specialized capabilities on demand.

### Why Skills?

1. **Context Efficiency**: Don't waste context on unused tool descriptions
2. **Specialization**: Skills can include detailed instructions for specific tasks
3. **Modularity**: Easy to add/remove capabilities
4. **Discoverability**: Agent can browse available skills

### SKILL.md Format

Skills are defined in markdown files with YAML frontmatter:

```markdown
---
name: skill-name
description: What this skill does
version: 1.0.0
tools:
  - tool1
  - tool2
---

# Skill Instructions

Detailed steps for how to use this skill...
```

In [25]:
# Let's look at the skills we created
skills_dir = Path("skills")

print("Available skills:")
for skill_dir in skills_dir.iterdir():
    if skill_dir.is_dir():
        skill_file = skill_dir / "SKILL.md"
        if skill_file.exists():
            content = skill_file.read_text()
            # Extract name and description from frontmatter
            lines = content.split("\n")
            name = ""
            desc = ""
            for line in lines:
                if line.startswith("name:"):
                    name = line.split(":", 1)[1].strip()
                if line.startswith("description:"):
                    desc = line.split(":", 1)[1].strip()
            print(f"  - {name}: {desc}")

Available skills:
  - meal-planning: Create personalized meal plans based on dietary needs and preferences
  - wellness-assessment: Assess user wellness goals and create personalized recommendations


In [26]:
# Read the wellness-assessment skill
skill_content = Path("skills/wellness-assessment/SKILL.md").read_text()
print(skill_content)

---
name: wellness-assessment
description: Assess user wellness goals and create personalized recommendations
version: 1.0.0
tools:
  - read_file
  - write_file
---

# Wellness Assessment Skill

You are conducting a comprehensive wellness assessment. Follow these steps:

## Step 1: Gather Information
Ask the user about:
- Current health goals (weight, fitness, stress, sleep)
- Any medical conditions or limitations
- Current exercise routine (or lack thereof)
- Dietary preferences and restrictions
- Sleep patterns and quality
- Stress levels and sources

## Step 2: Analyze Responses
Review the user's answers and identify:
- Primary wellness priority
- Secondary goals
- Potential barriers to success
- Existing healthy habits to build on

## Step 3: Create Assessment Report
Write a wellness assessment report to `workspace/wellness_assessment.md` containing:
- Summary of current wellness state
- Identified strengths
- Areas for improvement
- Recommended focus areas (prioritized)
- Suggeste

In [27]:
# Create a skill-aware tool
@tool
def load_skill(skill_name: str) -> str:
    """Load a skill's instructions for a specialized task.
    
    Available skills:
    - wellness-assessment: Assess user wellness and create recommendations
    - meal-planning: Create personalized meal plans
    
    Args:
        skill_name: Name of the skill to load
    
    Returns:
        Skill instructions
    """
    skill_path = Path(f"skills/{skill_name}/SKILL.md")
    if not skill_path.exists():
        available = [d.name for d in Path("skills").iterdir() if d.is_dir()]
        return f"Skill '{skill_name}' not found. Available: {', '.join(available)}"
    
    return skill_path.read_text()

print("Skill loader defined!")

Skill loader defined!


In [28]:
# Create an agent that can load and use skills
skill_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        load_skill,
        write_todos,
        update_todo,
        list_todos,
    ],
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a wellness assistant with access to specialized skills.

When a user asks for something that matches a skill:
1. Load the appropriate skill using load_skill()
2. Follow the skill's instructions carefully
3. Save outputs as specified in the skill

Available skills:
- wellness-assessment: For comprehensive wellness evaluations
- meal-planning: For creating personalized meal plans

If no skill matches, use your general wellness knowledge."""
)

print("Skill-aware agent created!")

Skill-aware agent created!


In [29]:
# Test with a skill-appropriate request
TODO_STORE.clear()

result = skill_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "I'd like a wellness assessment. I'm a 35-year-old office worker who sits most of the day, has trouble sleeping, and wants to lose 15 pounds. I'm vegetarian and have no major health conditions."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## Your Wellness Assessment Results

Based on your profile, I've identified that you have excellent foundational elements (vegetarian diet, health awareness, no major conditions) but face common modern lifestyle challenges. Here are my prioritized recommendations:

## Immediate Action Items (Start Today)
1. **Set Movement Reminders:** Use your phone/computer to remind you to stand and move for 2 minutes every hour
2. **Create a Sleep Wind-Down Routine:** Begin dimming lights and avoiding screens 1 hour before your intended bedtime
3. **Hydration Reset:** Start each day with a large glass of water and keep a water bottle at your desk

## Short-Term Goals (1-2 Weeks)
1. **Establish Desk Exercise Routine:** Implement 5-minute movement breaks every 2 hours (desk stretches, wall push-ups, calf raises)
2. **Sleep Schedule Consistency:** Go to bed and wake up at the same time every day, even weekends
3. **Protein-Rich Vegetarian Meals:** Plan meals around legumes, quinoa, tofu

## Task 9: Using deepagents-cli

The `deepagents-cli` provides an interactive terminal interface for working with Deep Agents.

### Installation

```bash
uv pip install deepagents-cli
# or
pip install deepagents-cli
```

### Key Features

| Feature | Description |
|---------|-------------|
| **Interactive Sessions** | Chat with your agent in the terminal |
| **Conversation Resume** | Pick up where you left off |
| **Human-in-the-Loop** | Approve or reject agent actions |
| **File System Access** | Agent can read/write to your filesystem |
| **Remote Sandboxing** | Run in isolated Docker containers |

### Basic Usage

```bash
# Start an interactive session
deepagents

# Resume a previous conversation
deepagents --resume

# Use a specific model
deepagents --model openai:gpt-4o

# Enable human-in-the-loop approval
deepagents --approval-mode full
```

### Example Session

```
$ deepagents

Welcome to Deep Agents CLI!

You: Create a 7-day meal plan for a vegetarian athlete

Agent: I'll create a comprehensive meal plan for you. Let me:
1. Research vegetarian athlete nutrition needs
2. Design balanced daily menus
3. Save the plan to a file

[Agent uses tools...]

Agent: I've created your meal plan! You can find it at:
workspace/vegetarian_athlete_meal_plan.md

You: /exit
```

In [30]:
# Check if CLI is installed
import subprocess

try:
    result = subprocess.run(["deepagents", "--version"], capture_output=True, text=True)
    print(f"deepagents-cli version: {result.stdout.strip()}")
except FileNotFoundError:
    print("deepagents-cli not installed. Install with:")
    print("  uv pip install deepagents-cli")
    print("  # or")
    print("  pip install deepagents-cli")

deepagents-cli not installed. Install with:
  uv pip install deepagents-cli
  # or
  pip install deepagents-cli


### Try It Yourself!

After installing the CLI, try these commands in your terminal:

```bash
# Basic interactive session
deepagents

# With a specific working directory
deepagents --workdir ./workspace

# See all options
deepagents --help
```

Sample prompts to try:
1. "Create a weekly workout plan and save it to a file"
2. "Research the health benefits of meditation and summarize in a report"
3. "Analyze my current diet and suggest improvements" (then provide details)

## Task 10: Building a Complete Deep Agent System

Now let's bring together all four elements to build a comprehensive "Wellness Coach" system:

1. **Planning**: Track multi-week wellness programs
2. **Context Management**: Store session notes and progress
3. **Subagent Spawning**: Delegate to specialists (exercise, nutrition, mindfulness)
4. **Long-term Memory**: Remember user preferences and history

In [31]:
# Define specialized wellness subagents
# Subagents inherit the backend from the parent, so they use the same workspace
exercise_specialist = {
    "name": "exercise-specialist",
    "description": "Expert in exercise science, workout programming, and physical fitness. Use for exercise-related questions and plan creation.",
    "system_prompt": """You are an exercise specialist with expertise in:
- Workout programming for different fitness levels
- Exercise form and safety
- Progressive overload principles
- Recovery and injury prevention

Always consider the user's fitness level and any physical limitations.
Provide clear, actionable exercise instructions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

nutrition_specialist = {
    "name": "nutrition-specialist",
    "description": "Expert in nutrition science, meal planning, and dietary optimization. Use for food-related questions and meal plans.",
    "system_prompt": """You are a nutrition specialist with expertise in:
- Macro and micronutrient balance
- Meal planning and preparation
- Dietary restrictions and alternatives
- Nutrition timing for performance

Always respect dietary restrictions and preferences.
Focus on practical, achievable meal suggestions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

mindfulness_specialist = {
    "name": "mindfulness-specialist",
    "description": "Expert in stress management, sleep optimization, and mental wellness. Use for stress, sleep, and mental health questions.",
    "system_prompt": """You are a mindfulness and mental wellness specialist with expertise in:
- Stress reduction techniques
- Sleep hygiene and optimization
- Meditation and breathing exercises
- Work-life balance strategies

Be supportive and non-judgmental.
Provide practical techniques that can be implemented immediately.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Specialist subagents defined!")

Specialist subagents defined!


In [32]:
# Create the Wellness Coach coordinator
wellness_coach = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        # Planning
        write_todos,
        update_todo,
        list_todos,
        # Long-term Memory
        get_user_profile,
        save_user_preference,
        # Skills
        load_skill,
    ],
    backend=filesystem_backend,  # All file ops go to workspace
    subagents=[exercise_specialist, nutrition_specialist, mindfulness_specialist],
    system_prompt="""You are a Personal Wellness Coach that coordinates comprehensive wellness programs.

## Your Role
- Understand each user's unique goals, constraints, and preferences
- Create personalized, multi-week wellness programs
- Coordinate between exercise, nutrition, and mindfulness specialists
- Track progress and adapt recommendations

## Workflow
1. **Initial Assessment**: Get user profile and understand their situation
2. **Planning**: Create a todo list for the program components
3. **Delegation**: Use specialists for domain-specific content:
   - exercise-specialist: Workout plans and fitness guidance
   - nutrition-specialist: Meal plans and dietary advice
   - mindfulness-specialist: Stress and sleep optimization
4. **Integration**: Combine specialist outputs into a cohesive program
5. **Documentation**: Save all plans and recommendations to files

## Important
- Always check user profile first for context
- Respect any medical conditions or dietary restrictions
- Provide clear, actionable recommendations
- Save progress to files so users can reference later"""
)

print("Wellness Coach created with all 4 Deep Agent elements!")

Wellness Coach created with all 4 Deep Agent elements!


In [33]:
# Test the complete system
TODO_STORE.clear()

result = wellness_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I'd like you to create a 2-week wellness program for me.

I want to focus on:
1. Building a consistent exercise routine (I can exercise 3x per week for 30 mins)
2. Improving my diet (remember I'm vegetarian)
3. Better managing my work stress and improving my sleep

Please create comprehensive plans for each area and save them as separate files I can reference."""
    }]
})

print("Wellness Coach response:")
print(result["messages"][-1].content)

Wellness Coach response:
Perfect, Alex! 🎉 I've created your comprehensive 2-week wellness program with all the components you requested. Here's what I've delivered:

## Your Complete Wellness Program Files:

### 📋 **Master Program Overview**
- **File:** `/alex_master_wellness_program.txt`
- Complete coordination guide with all program components
- Quick reference emergency protocols
- Progress tracking system
- Success tips tailored to your profile

### 💪 **Exercise Program**  
- **File:** `/exercise_program/alex_2_week_workout_plan.txt`
- 3x/week, 30-minute morning workouts
- Week 1: Foundation building (gentle strength, relaxing yoga, low-impact cardio)
- Week 2: Progressive enhancement 
- All exercises chosen for anxiety management and energy building

### 🥗 **Vegetarian Nutrition Plan**
- **File:** `/nutrition_plan_for_alex.txt` 
- 14 days of complete meal plans with snacks
- Focus on energy-boosting and sleep-supporting foods
- Grocery lists and meal prep tips included
- Anxiety-f

In [34]:
# Review what was created
print("=" * 60)
print("FINAL TODO STATUS")
print("=" * 60)
print(list_todos.invoke({}))

print("\n" + "=" * 60)
print("GENERATED FILES")
print("=" * 60)
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

FINAL TODO STATUS
✅ [todo_1] Create comprehensive exercise plan (completed)
✅ [todo_3] Develop vegetarian nutrition plan (completed)
✅ [todo_5] Design stress management and sleep optimization plan (completed)
✅ [todo_7] Create integrated daily schedules (completed)
✅ [todo_9] Compile master wellness program document (completed)

GENERATED FILES
  [FILE] actionable_sleep_plan.md (6977 bytes)
  [FILE] alex_2week_integrated_daily_schedules.txt (6626 bytes)
  [FILE] alex_master_wellness_program.txt (6902 bytes)
  [FILE] comprehensive_morning_routine_guide.md (19615 bytes)
  [FILE] comprehensive_sleep_research_summary.md (20744 bytes)
  [FILE] evidence_based_stress_management_guide.md (13741 bytes)
  [FILE] Evidence_Based_Stress_Management_Report.md (20303 bytes)
  [DIR] exercise_program/
  [FILE] morning_routine_energy_guide.md (67791 bytes)
  [FILE] my_sleep_improvement_plan.md (7420 bytes)
  [FILE] nutrition_plan_for_alex.txt (4554 bytes)
  [FILE] personalized_sleep_improvement_plan.md (

In [35]:
# Read one of the generated files
files = list(WORKSPACE.glob("*.md"))
if files:
    print(f"\nContents of {files[0].name}:")
    print("=" * 60)
    print(files[0].read_text()[:2000] + "..." if len(files[0].read_text()) > 2000 else files[0].read_text())


Contents of actionable_sleep_plan.md:
# ACTIONABLE SLEEP IMPROVEMENT PLAN
## Evidence-Based Solutions for Your Specific Issues

---

## ðŸŽ¯ PRIORITY ACTION PLAN

### **WEEK 1-2: FOUNDATION BUILDING**

#### Day 1 Actions:
1. **Choose Your Fixed Wake Time**
   - Select one wake time for ALL days (including weekends)
   - Recommended: 6:30-7:30 AM based on your current 10pm-1am bedtime range
   - Set this time and DO NOT vary it by more than 30 minutes

2. **Create Phone-Free Bedroom**
   - Move charger outside bedroom immediately
   - Buy analog alarm clock ($10-20)
   - If phone needed for emergencies, use airplane mode + alarm only

3. **Bedroom Environment Optimization**
   - Set thermostat to 65Â°F (18Â°C) for sleep
   - Install blackout curtains or use eye mask
   - Remove all LED lights and electronic displays

#### Days 2-14 Protocol:

**MORNING ROUTINE (First 30 minutes awake):**
- Get bright light exposure IMMEDIATELY (step outside or near window)
- Drink 16-24 oz water
- Do 5

---
## ❓ Question #3:

What are the key considerations when designing **subagent configurations**?

Consider:
- When should subagents share tools vs have distinct tools?
- How do you decide which model to use for each subagent?
- What's the right granularity for subagent specialization?

##### Answer:
When designing subagent configurations, the biggest thing is being clear about responsibility. In the notebook, we saw how tools and roles were explicitly defined. Subagents should follow the same idea — each one should have a clear purpose instead of doing everything.

Subagents should share tools only when it makes sense. For example, file tools or todo tracking tools could be shared if multiple agents need to read or write files. But if a subagent has a very specific job (like only doing research or only generating reports), then giving it only the tools it actually needs keeps things cleaner and safer. Too many shared tools can make behavior unpredictable.

Choosing the model for each subagent depends on complexity and cost. If a subagent is doing heavy reasoning or research, you might use a stronger model. If it’s doing simple formatting or summarization, a smaller and cheaper model is enough. The notebook showed how model configuration is explicit, so in production you would be intentional about which model each subagent uses.

As for granularity, subagents shouldn’t be too broad or too tiny. If they’re too broad, you lose specialization. If they’re too narrow, you end up with too many agents and coordination overhead. The right balance is when each subagent owns one meaningful responsibility, like “research”, “planning”, or “report writing.”

## ❓ Question #4:

For a **production wellness application** using Deep Agents, what would you need to add?

Consider:
- Safety guardrails for health advice
- Persistent storage (not in-memory)
- Multi-user support and isolation
- Monitoring and observability
- Cost management with subagents

##### Answer:
For a production wellness application, what we built in the notebook would not be enough. It’s a good demo, but production needs more layers.

First, safety guardrails are critical. Since it’s health-related, the system must clearly avoid giving medical diagnosis or unsafe advice. There should be rules in the system prompt and possibly validation layers to prevent harmful outputs.

Second, persistent storage is necessary. In the notebook, todos were stored in memory using TODO_STORE, which resets every time. In production, you would need a real database so user history, progress, and conditions don’t disappear.

Third, multi-user support and isolation would be required. Right now, everything runs in a shared workspace. In production, each user would need isolated storage and separate state so their data doesn’t mix with others.

Monitoring and observability are also important. You would want logging of tool calls, errors, model usage, and workflow steps so you can debug and improve the system. In the notebook, we manually printed outputs, but in production that would be structured logging.

Finally, cost management becomes very important. If you use multiple subagents or strong models everywhere, costs can grow fast. So you would need strategies like using cheaper models for simple tasks, limiting retries, and monitoring token usage.

So overall, the notebook gives us the architecture basics — tools, todos, file systems, agents — but production requires safety, persistence, isolation, monitoring, and cost control on top of that foundation.

---
## 🏗️ Activity #2: Build a Wellness Coach Agent

Build your own wellness coach that uses all 4 Deep Agent elements.

### Requirements:
1. **Planning**: Create todos for a 30-day wellness challenge
2. **Context Management**: Store daily check-in notes
3. **Subagents**: At least 2 specialized subagents
4. **Memory**: Remember user preferences across interactions

### Challenge:
Create a "30-Day Wellness Challenge" system that:
- Generates a personalized 30-day plan
- Tracks daily progress
- Adapts recommendations based on feedback
- Saves a weekly summary report

In [ ]:
### YOUR CODE HERE ###

# Step 1: Define your subagent configurations


# Step 2: Create any additional tools you need


# Step 3: Build the main coordinator agent


# Step 4: Test with a user creating their 30-day challenge


# Step 5: Simulate a daily check-in and adaptation


In [ ]:
### YOUR CODE HERE ###

# Step 0: Reset todo store
TODO_STORE.clear()


# Step 1: Define your subagent configurations
# (Following exact structure from Task 6 & Task 10 in notebook)

exercise_specialist = {
    "name": "exercise-specialist",
    "description": "Use for exercise programming and workout planning.",
    "system_prompt": """You are an exercise specialist.

Create safe, structured workout plans.
Consider user preferences and limitations.
Be practical and actionable.""",
    "tools": [],
    "model": "openai:gpt-4o-mini",
}

mindfulness_specialist = {
    "name": "mindfulness-specialist",
    "description": "Use for stress management and mental wellness guidance.",
    "system_prompt": """You are a mindfulness specialist.

Provide stress reduction techniques,
sleep support strategies,
and emotional wellness practices.

Be supportive and clear.""",
    "tools": [],
    "model": "openai:gpt-4o-mini",
}


# Step 2: No new tools required (Reusing notebook tools)
# Planning → write_todos, update_todo, list_todos
# Memory → get_user_profile, save_user_preference
# File handling → built-in via backend


# Step 3: Build the main coordinator agent

wellness_challenge_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        write_todos,
        update_todo,
        list_todos,
        get_user_profile,
        save_user_preference,
    ],
    backend=filesystem_backend,
    subagents=[exercise_specialist, mindfulness_specialist],
    system_prompt="""You are a 30-Day Wellness Challenge Coordinator.

Your responsibilities:

1. Check user profile at the beginning.
2. Create todos for the 30-day challenge.
3. Delegate:
   - exercise planning → exercise-specialist
   - stress & sleep support → mindfulness-specialist
4. Save plans as markdown files.
5. Store daily check-ins as files.
6. Adapt recommendations based on feedback.
7. Save weekly summary reports.

Be structured and organized."""
)

print("30-Day Wellness Challenge Agent created!")


# Step 4: Test with a user creating their 30-day challenge

result = wellness_challenge_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex.

I want to start a 30-day wellness challenge.
My goals are:
- Exercise 3 times per week
- Reduce stress
- Improve sleep consistency

Please create a personalized 30-day challenge plan and save it to files."""
    }]
})

print("Initial Challenge Response:")
print(result["messages"][-1].content)


print("\nTodo Status After Plan Creation:")
print(list_todos.invoke({}))


# Step 5: Simulate a daily check-in and adaptation

result_checkin = wellness_challenge_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Daily Check-in:
Day 3 completed.
I skipped one workout but practiced breathing exercises.
Feeling slightly stressed due to work.

Please adjust my plan if needed and save today's note."""
    }]
})

print("\nDaily Check-in Response:")
print(result_checkin["messages"][-1].content)


print("\nFinal Todo Status:")
print(list_todos.invoke({}))


print("\nGenerated Files:")
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"[FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"[DIR] {f.name}/")


30-Day Wellness Challenge Agent created!
Initial Challenge Response:
Perfect! Your personalized 30-Day Wellness Challenge is now completely set up! 🎉

## Here's what I've created for you:

### **Core Challenge Files:**
1. **Master Plan** (`/30_day_wellness_challenge_master_plan.md`) - Your complete challenge overview
2. **Exercise Plan** (`/exercise_plan_30_days.md`) - Detailed 3x/week workout progression 
3. **Stress & Sleep Plan** (`/stress_sleep_improvement_plan.md`) - Daily routines for anxiety management and better sleep
4. **Daily Check-in Template** (`/daily_checkin_template.md`) - Track your daily progress
5. **Weekly Review Template** (`/weekly_review_template.md`) - Assess and adjust weekly

### **Your Personalized Challenge Features:**
- **Morning-focused workouts** (7:00-8:00 AM) aligned with your preference
- **Anxiety-friendly stress management** techniques designed for mild anxiety
- **Vegetarian-supportive** exercise and wellness recommendations
- **Detailed guidance** 

## Activity #2 Summary

In this activity, we built a complete 30-Day Wellness Challenge system using all four Deep Agent elements: planning, context management, subagents, and long-term memory. The agent created explicit todos for the challenge, delegated exercise and stress-related planning to specialized subagents, stored daily check-ins as files, and adapted recommendations based on user feedback. It also retrieved and saved user preferences to ensure personalization across interactions.

This activity demonstrated how Deep Agents can coordinate multiple components — planning, memory, delegation, and file handling — to build a structured and evolving wellness program instead of just generating static advice.

As you can see in the output, todos were created and updated over time, personalized plan files were generated in the workspace, and the system adapted recommendations after the daily check-in, confirming that all four Deep Agent elements were implemented correctly.

---
## Summary

In this session, we explored **Deep Agents** and their four key elements:

| Element | Purpose | Implementation |
|---------|---------|----------------|
| **Planning** | Track complex tasks | `write_todos`, `update_todo`, `list_todos` |
| **Context Management** | Handle large contexts | File system tools, automatic offloading |
| **Subagent Spawning** | Delegate to specialists | `task` tool with custom configs |
| **Long-term Memory** | Remember across sessions | LangGraph Store integration |

### Key Takeaways:

1. **Deep Agents handle complexity** - Unlike simple tool loops, they can manage long-horizon, multi-step tasks
2. **Planning is context engineering** - Todo lists and files aren't just organization—they're extended memory
3. **Subagents prevent context bloat** - Delegation keeps the main agent focused and efficient
4. **Skills enable progressive disclosure** - Load capabilities on-demand instead of upfront
5. **The CLI makes interaction natural** - Interactive sessions with conversation resume

### Deep Agents vs Traditional Agents

| Aspect | Traditional Agent | Deep Agent |
|--------|-------------------|------------|
| Task complexity | Simple, single-step | Complex, multi-step |
| Context management | All in conversation | Files + summaries |
| Delegation | None | Subagent spawning |
| Memory | Within thread | Across sessions |
| Planning | Implicit | Explicit (todos) |

### When to Use Deep Agents

**Use Deep Agents when:**
- Tasks require multiple steps or phases
- Context would overflow in a simple loop
- Specialization would improve quality
- Users need to resume sessions
- Long-term memory is valuable

**Use Simple Agents when:**
- Tasks are straightforward Q&A
- Single tool call suffices
- Context fits easily
- No need for persistence

### Further Reading

- [Deep Agents Documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents GitHub](https://github.com/langchain-ai/deepagents)
- [Context Management Blog Post](https://www.blog.langchain.com/context-management-for-deepagents/)
- [Building Multi-Agent Applications](https://www.blog.langchain.com/building-multi-agent-applications-with-deep-agents/)
- [LangGraph Memory Concepts](https://langchain-ai.github.io/langgraph/concepts/memory/)